In [9]:
import pandas as pd

# Import Raw Data
raw_df = pd.read_csv('superstore/raw/sample_superstore.csv')
print(raw_df.head())

   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
1       2  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
2       3  CA-2016-138688   6/12/2016   6/16/2016    Second Class    DV-13045   
3       4  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   
4       5  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  Postal Code  Region       Product ID         Category Sub-Category  \
0       42420   Sout

In [5]:
# Check for Nulls
null_counts = raw_df.isnull().sum()
print(null_counts)

Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64


In [11]:
# Small Caps and Space replacing with underscore
raw_df.columns = [col.lower().replace(' ', '_').replace('-', '_') for col in raw_df.columns]
print(raw_df.head())

   row_id        order_id  order_date   ship_date       ship_mode customer_id  \
0       1  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
1       2  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
2       3  CA-2016-138688   6/12/2016   6/16/2016    Second Class    DV-13045   
3       4  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   
4       5  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   

     customer_name    segment        country             city  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  United States      Los Angeles  ...   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale  ...   

  postal_code  region       product_id         category sub_category  \
0       42420   Sout

In [55]:
# Convert date to datetime format and normal format
raw_df['order_date'] = pd.to_datetime(raw_df['order_date'])
raw_df['ship_date'] = pd.to_datetime(raw_df['ship_date'])
# Create a 'month_year' column 
# 'M' creates a Period object (e.g., 2016-11)
raw_df['month_year'] = raw_df['order_date'].dt.to_period('M')
# Group by month_year and sum Sales and Profit
# same as: SELECT month_year, SUM(sales), SUM(profit) FROM table GROUP BY month_year
monthly_performance = raw_df.groupby('month_year').agg(
    total_sales=('sales', 'sum'),
    total_profit=('profit', 'sum')).reset_index()
# Calculate Profit Margin percentage
monthly_performance['profit_margin_%'] = (monthly_performance['total_profit'] / monthly_performance['total_sales']) * 100
# Round to two decimal places
monthly_performance = monthly_performance.round(2)
# Sort by date to ensure the timeline is correct
monthly_performance = monthly_performance.sort_values('month_year')
print("--- Monthly Summary ---")
print(monthly_performance.tail(12))

--- Monthly Summary ---
   month_year  total_sales  total_profit  profit_margin_%
36    2017-01     43971.37       7140.44            16.24
37    2017-02     20301.13       1613.87             7.95
38    2017-03     58872.35      14751.89            25.06
39    2017-04     36521.54        933.29             2.56
40    2017-05     44261.11       6342.58            14.33
41    2017-06     52981.73       8223.34            15.52
42    2017-07     45264.42       6952.62            15.36
43    2017-08     63120.89       9040.96            14.32
44    2017-09     87866.65      10991.56            12.51
45    2017-10     77776.92       9275.28            11.93
46    2017-11    118447.82       9690.10             8.18
47    2017-12     83829.32       8483.35            10.12


In [53]:
# Group by Customer and aggregate both Sales and Profit
customer_stats = raw_df.groupby(['customer_id', 'customer_name']).agg(
    total_revenue=('sales', 'sum'),
    total_profit=('profit', 'sum')).reset_index()
# Calculate Profit Margin percentage
customer_stats['profit_margin_%'] = (customer_stats['total_profit'] / customer_stats['total_revenue']) * 100
# Sort by Revenue to find the top 10
top_10_combined = customer_stats.sort_values(by='total_revenue', ascending=False).head(10)
# Round all numeric columns to 2 decimal places
top_10_combined = top_10_combined.round(2)
# Display the final result
print("--- Top 10 Customers: Revenue vs. Profit ---")
print(top_10_combined)

--- Top 10 Customers: Revenue vs. Profit ---
    customer_id       customer_name  total_revenue  total_profit  \
700    SM-20320         Sean Miller       25043.05      -1980.74   
741    TC-20980        Tamara Chand       19052.22       8981.32   
621    RB-19360        Raymond Buch       15117.34       6976.10   
730    TA-21385        Tom Ashbrook       14595.62       4703.79   
6      AB-10105       Adrian Barton       14473.57       5444.81   
434    KL-16645        Ken Lonsdale       14175.23        806.86   
669    SC-20095        Sanjit Chand       14142.33       5757.41   
327    HL-15040        Hunter Lopez       12873.30       5622.43   
683    SE-20110        Sanjit Engle       12209.44       2650.68   
131    CC-12370  Christopher Conant       12129.07       2177.05   

     profit_margin_%  
700            -7.91  
741            47.14  
621            46.15  
730            32.23  
6              37.62  
434             5.69  
669            40.71  
327            43.68  

In [51]:
# Group by Category and Sub-Category
category_analysis = raw_df.groupby(['category', 'sub_category','product_name']).agg(
    total_sales=('sales', 'sum'),
    total_profit=('profit', 'sum')).reset_index()
# Filter for loss-making sub-categories (where profit is less than 0)
losses = category_analysis[category_analysis['total_profit'] < 0]
# Sort by total_profit in ascending order (most negative at the top)
losses = losses.sort_values(by='total_profit', ascending=True)
# Final Formatting
losses = losses.round(2)
print("--- Loss-Making Categories/Sub-Categories ---")
print(losses.head())

--- Loss-Making Categories/Sub-Categories ---
        category sub_category  \
1613  Technology     Machines   
1632  Technology     Machines   
1614  Technology     Machines   
352    Furniture       Tables   
346    Furniture       Tables   

                                           product_name  total_sales  \
1613          Cubify CubeX 3D Printer Double Head Print     11099.96   
1632          Lexmark MX611dhe Monochrome Laser Printer     16829.90   
1614          Cubify CubeX 3D Printer Triple Head Print      7999.98   
352   Chromcraft Bull-Nose Wood Oval Conference Tabl...      9917.64   
346   Bush Advantage Collection Racetrack Conference...      9544.72   

      total_profit  
1613      -8879.97  
1632      -4589.97  
1614      -3839.99  
352       -2876.12  
346       -1934.40  


In [49]:
# Group by Region and aggregate multiple metrics
region_performance = raw_df.groupby('region').agg(
    total_sales=('sales', 'sum'),
    total_profit=('profit', 'sum'),
    order_count=('order_id', 'nunique')).reset_index()
# Calculate Profit Margin percentage for each region
region_performance['profit_margin_%'] = (region_performance['total_profit'] / region_performance['total_sales']) * 100
# Sort by Profit Margin to see the most efficient regions first
region_performance = region_performance.sort_values(by='profit_margin_%', ascending=False)
# Round values to 2 decimal places
region_performance = region_performance.round(2)
# Display the performance table
print("--- Region-Wise Performance Comparison ---")
print(region_performance)

--- Region-Wise Performance Comparison ---
    region  total_sales  total_profit  order_count  profit_margin_%
3     West    725457.82     108418.45         1611            14.94
1     East    678781.24      91522.78         1401            13.48
2    South    391721.90      46749.43          822            11.93
0  Central    501239.89      39706.36         1175             7.92


In [47]:
# Group by Customer and find the minimum (earliest) date
customer_first_purchase = raw_df.groupby(['customer_id', 'customer_name'])['order_date'].min().reset_index()
# Rename the column to be clear
customer_first_purchase = customer_first_purchase.rename(columns={'order_date': 'first_purchase_date'})
# Sort by the date to see your oldest customers at the top
customer_first_purchase = customer_first_purchase.sort_values(by='first_purchase_date')
# Display the result
print("--- Customer Acquisition Dates ---")
print(customer_first_purchase.head())

--- Customer Acquisition Dates ---
    customer_id  customer_name first_purchase_date
229    DP-13000  Darren Powers          2014-01-03
605    PO-19195  Phillina Ober          2014-01-04
485    MB-18085     Mick Brown          2014-01-05
396    JO-15145  Jack O'Briant          2014-01-06
497    ME-17320  Maria Etezadi          2014-01-06


In [45]:
# Count unique orders per customer
customer_order_counts = raw_df.groupby('customer_id')['order_id'].nunique().reset_index()
customer_order_counts.columns = ['customer_id', 'order_count']
# Identify total unique customers
total_customers = customer_order_counts.shape[0]
# Identify repeat customers (those with more than 1 order)
repeat_customers = customer_order_counts[customer_order_counts['order_count'] > 1].shape[0]
# Calculate the rate
repeat_rate = (repeat_customers / total_customers) * 100
# Display the findings
print(f"Total Customers: {total_customers}")
print(f"Repeat Customers: {repeat_customers}")
print(f"Repeat Purchase Rate: {repeat_rate:.2f}%")

Total Customers: 793
Repeat Customers: 781
Repeat Purchase Rate: 98.49%


In [25]:
# Aggregate basic metrics per customer
clv_df = raw_df.groupby(['customer_id', 'customer_name']).agg(
    total_revenue=('sales', 'sum'),
    total_profit=('profit', 'sum'),
    order_count=('order_id', 'nunique'),
    first_purchase=('order_date', 'min'),
    last_purchase=('order_date', 'max')).reset_index()
# Calculate Customer Lifespan in days
clv_df['lifespan_days'] = (clv_df['last_purchase'] - clv_df['first_purchase']).dt.days
# Calculate Average Order Value (AOV)
clv_df['avg_order_value'] = clv_df['total_revenue'] / clv_df['order_count']
# Define CLV (using Profit as the value metric)
# Rename 'total_profit' to 'historical_clv' for clarity
clv_df = clv_df.rename(columns={'total_profit': 'historical_clv'})
# Sort by CLV to see most valuable customers
top_clv = clv_df.sort_values(by='historical_clv', ascending=False).head(10)
# Formatting
print("\n--- Top 10 Customers by Historical CLV (Profit) ---")
print(top_clv[['customer_name', 'order_count', 'lifespan_days', 'avg_order_value', 'historical_clv']].round(2))


--- Top 10 Customers by Historical CLV (Profit) ---
            customer_name  order_count  lifespan_days  avg_order_value  \
741          Tamara Chand            5            750          3810.44   
621          Raymond Buch            6            542          2519.56   
669          Sanjit Chand            9           1068          1571.37   
327          Hunter Lopez            6           1397          2145.55   
6           Adrian Barton           10           1065          1447.36   
730          Tom Ashbrook            4           1136          3648.90   
160  Christopher Martinez            4            983          2238.50   
424         Keith Dawkins           12           1077           681.77   
48            Andy Reiter            6           1125          1101.41   
234         Daniel Raglin            8           1323          1043.86   

     historical_clv  
741         8981.32  
621         6976.10  
669         5757.41  
327         5622.43  
6           5444.81  


In [57]:
# 1. Prepare Monthly Data
monthly_growth = raw_df.groupby('month_year')['sales'].sum().reset_index()
monthly_growth = monthly_growth.sort_values('month_year')
# 2. Month-over-Month (MoM) Comparison
# .shift(1) looks at the row exactly above it
monthly_growth['prev_month_sales'] = monthly_growth['sales'].shift(1)
monthly_growth['mom_growth_%'] = ((monthly_growth['sales'] - monthly_growth['prev_month_sales']) / monthly_growth['prev_month_sales']) * 100
# 3. Year-over-Year (YoY) Comparison
# Since we have 12 months in a year, we shift by 12
monthly_growth['prev_year_sales'] = monthly_growth['sales'].shift(12)
monthly_growth['yoy_growth_%'] = ((monthly_growth['sales'] - monthly_growth['prev_year_sales']) / monthly_growth['prev_year_sales']) * 100

print("--- Time-Series Growth Analysis ---")
# Showing the last year of data
print(monthly_growth.tail(12).round(2)) 

--- Time-Series Growth Analysis ---
   month_year      sales  prev_month_sales  mom_growth_%  prev_year_sales  \
36    2017-01   43971.37          96999.04        -54.67         18542.49   
37    2017-02   20301.13          43971.37        -53.83         22978.82   
38    2017-03   58872.35          20301.13        190.00         51715.88   
39    2017-04   36521.54          58872.35        -37.96         38750.04   
40    2017-05   44261.11          36521.54         21.19         56987.73   
41    2017-06   52981.73          44261.11         19.70         40344.53   
42    2017-07   45264.42          52981.73        -14.57         39261.96   
43    2017-08   63120.89          45264.42         39.45         31115.37   
44    2017-09   87866.65          63120.89         39.20         73410.02   
45    2017-10   77776.92          87866.65        -11.48         59687.74   
46    2017-11  118447.82          77776.92         52.29         79411.97   
47    2017-12   83829.32         118447.

In [59]:
# Calculate Average Profit per Category
cat_perf = raw_df.groupby('category')['profit'].mean().reset_index()
# Get the Overall Average of all categories
overall_avg_profit = cat_perf['profit'].mean()
# Compare Category vs Overall
cat_perf['overall_avg_profit'] = overall_avg_profit
cat_perf['profit_diff'] = cat_perf['profit'] - overall_avg_profit
cat_perf['performance_ratio'] = ((cat_perf['profit'] - overall_avg_profit) / overall_avg_profit).round(2)
# Sort by Sales in Descending Order
cat_perf = cat_perf.sort_values(by='profit', ascending=False)
print("--- Category vs. Global Average ---")
print(cat_perf.round(2))

--- Category vs. Global Average ---
          category  profit  overall_avg_profit  profit_diff  performance_ratio
2       Technology   78.75               35.93        42.83               1.19
1  Office Supplies   20.33               35.93       -15.60              -0.43
0        Furniture    8.70               35.93       -27.23              -0.76


In [61]:
# Group by Segment to get core metrics
segment_analysis = raw_df.groupby('segment').agg(
    total_sales=('sales', 'sum'),
    total_profit=('profit', 'sum'),
    order_count=('order_id', 'nunique')).reset_index()
# Calculate Profit Margin and Average Order Value
segment_analysis['profit_margin_%'] = (segment_analysis['total_profit'] / segment_analysis['total_sales']) * 100
segment_analysis['avg_order_value'] = segment_analysis['total_sales'] / segment_analysis['order_count']
# Calculate the Global Benchmark (Company-wide Margin)
overall_margin = (raw_df['profit'].sum() / raw_df['sales'].sum()) * 100
# Identify Underperformers (Below overall average margin)
underperformers = segment_analysis[segment_analysis['profit_margin_%'] < overall_margin].copy()
# Sort by lowest margin first
underperformers = underperformers.sort_values(by='profit_margin_%', ascending=True)
# Display Results
print(f"Company Benchmark Margin: {overall_margin:.2f}%")
print("--- Underperforming Segments ---")
print(segment_analysis.round(2))

Company Benchmark Margin: 12.47%
--- Underperforming Segments ---
       segment  total_sales  total_profit  order_count  profit_margin_%  \
0     Consumer   1161401.34     134119.21         2586            11.55   
1    Corporate    706146.37      91979.13         1514            13.03   
2  Home Office    429653.15      60298.68          909            14.03   

   avg_order_value  
0           449.11  
1           466.41  
2           472.67  
